In [1]:
import duckdb
import pandas as pd

db_path = 'database/analytics.duckdb'

with duckdb.connect(db_path) as conn:
    print("=== Таблицы в базе данных ===\n")
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchdf()
    print(tables.to_string(index=False))

    for table in tables['name']:
        print(f"\n\n=== Таблица: {table} ===")
        # Схема
        schema = conn.execute(f"DESCRIBE {table}").fetchdf()
        print("\nСтруктура:\n", schema.to_string(index=False))
        # Количество записей
        count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"\nКоличество записей: {count}")
        # Пример данных
        sample = conn.execute(f"SELECT * FROM {table} LIMIT 2").fetchdf()
        print("\nПример данных (первые 2 стр):\n", sample.to_string(index=False))

=== Таблицы в базе данных ===

       name
  consumers
    markets
order_cards
parser_data
   products


=== Таблица: consumers ===

Структура:
 column_name column_type null key                    default extra
         id      BIGINT   NO PRI nextval('consumer_id_seq')  None
       name     VARCHAR   NO UNI                        NaN  None
     eis_id     VARCHAR  YES NaN                        NaN  None

Количество записей: 4694

Пример данных (первые 2 стр):
  id                                                                                                                                       name eis_id
  1 ЛЕНИНГРАДСКОЕ ОБЛАСТНОЕ ГОСУДАРСТВЕННОЕ БЮДЖЕТНОЕ УЧРЕЖДЕНИЕ "ПОДПОРОЖСКИЙ СОЦИАЛЬНО-РЕАБИЛИТАЦИОННЫЙ ЦЕНТР ДЛЯ НЕСОВЕРШЕННОЛЕТНИХ "СЕМЬЯ" 793013
  2 ГОСУДАРСТВЕННОЕ БЮДЖЕТНОЕ ОБЩЕОБРАЗОВАТЕЛЬНОЕ УЧРЕЖДЕНИЕ СРЕДНЯЯ ОБЩЕОБРАЗОВАТЕЛЬНАЯ ШКОЛА №195 КРАСНОГВАРДЕЙСКОГО РАЙОНА САНКТ-ПЕТЕРБУРГА 627079


=== Таблица: markets ===

Структура:
 column_name column_type null key           

In [2]:
with duckdb.connect(db_path) as conn:
    # Общее количество записей
    total = conn.execute("SELECT COUNT(*) FROM consumers").fetchone()[0]
    print(f"\nВсего записей в таблице consumers: {total}")
    
    # Количество Null в eis_id
    null_count = conn.execute("SELECT COUNT(*) FROM consumers WHERE eis_id IS NULL").fetchone()[0]
    print(f"Записей с NULL в eis_id: {null_count}")
    # Найти все строки, где eis_id не NULL и содержит нецифровые символы
    query = """
        SELECT * 
        FROM consumers 
        WHERE eis_id IS NOT NULL 
          AND regexp_matches(eis_id, '[^0-9]');
    """
        
    df_invalid = conn.execute(query).fetchdf()
    print(f"Записи с eis_id, содержащими нецифровые символы: {len(df_invalid['eis_id'])}")
    print(df_invalid)


Всего записей в таблице consumers: 4694
Записей с NULL в eis_id: 0
Записи с eis_id, содержащими нецифровые символы: 0
Empty DataFrame
Columns: [id, name, eis_id]
Index: []


In [3]:
with duckdb.connect(db_path) as conn:
    # проверка на пустые значения в order_cards, будут заполнены на следующем этапе парсинга
    query = """
        SELECT
            COUNT(*) AS rows_amount,
            COUNT(*) - COUNT(contract_number) AS null_contract_number,
            COUNT(*) - COUNT(contract_type)   AS null_contract_type,
            COUNT(*) - COUNT(placement_date)  AS null_placement_date,
            COUNT(*) - COUNT(end_date)        AS null_end_date,
            COUNT(*) - COUNT(update_date)     AS null_update_date,
            COUNT(*) - COUNT(name)            AS null_name,
            COUNT(*) - COUNT(consumer_id)     AS null_consumer_id,
            COUNT(*) - COUNT(total_price)     AS null_total_price,
            COUNT(*) - COUNT(status)          AS null_status
        FROM order_cards;
    """
    resp = conn.execute(query).fetchdf()
    # Транспонируем для удобного отображения
    null_counts = resp.T.reset_index()
    null_counts.columns = ['column', 'null_count']   
    print("=== Количество NULL в столбцах order_cards ===\n")
    print(null_counts.to_string(index=False))

=== Количество NULL в столбцах order_cards ===

              column  null_count
         rows_amount      597524
null_contract_number           0
  null_contract_type           0
 null_placement_date           0
       null_end_date       26985
    null_update_date           0
           null_name          63
    null_consumer_id           0
    null_total_price           1
         null_status           0


In [4]:
# проверка количества отпарсенных карточек в order_cards с зарегистрированным количеством в parser_data
with duckdb.connect(db_path) as conn:
    # Зарегистрированное количество (из parser_data)
    reg = conn.execute("SELECT SUM(card_parsed) as registered FROM parser_data WHERE is_parsed = true").fetchdf()
    # Фактическое количество записей в order_cards
    fact = conn.execute("SELECT COUNT(*) as actual FROM order_cards").fetchdf()
    
    registered = reg['registered'].iloc[0]
    actual = fact['actual'].iloc[0]
    
    print(f"Зарегистрировано контрактов (по parser_data): {registered}")
    print(f"Фактически в order_cards: {actual}")
    is_discrepancy = True
    if registered == actual:
        print("Количество совпадает.")
        is_discrepancy = False
    else:
        print(f"Расхождение: {registered - actual} записей (зарегистрировано больше, чем в БД).")

Зарегистрировано контрактов (по parser_data): 598189.0
Фактически в order_cards: 597524
Расхождение: 665.0 записей (зарегистрировано больше, чем в БД).


In [5]:
# Если есть расхождение ищем даты, которые надо отпарсить повторно
if is_discrepancy:
    with duckdb.connect(db_path) as conn:
        query4 = """
            SELECT
                pd.date,
                pd.card_parsed AS registered,
                COALESCE(oc.cnt, 0) AS actual,
                pd.card_parsed - COALESCE(oc.cnt, 0) AS diff
            FROM parser_data pd
            LEFT JOIN (
                SELECT placement_date, COUNT(*) AS cnt
                FROM order_cards
                GROUP BY placement_date
            ) oc ON pd.date = oc.placement_date
            WHERE pd.is_parsed = TRUE
              AND pd.card_parsed != COALESCE(oc.cnt, 0);
        """
        diff_df = conn.execute(query4).fetchdf()
        
        print("=== Даты с расхождениями ===\n")
        if diff_df.empty:
            print("✅ Нет расхождений между card_parsed и количеством записей в order_cards.")
        else:
            # print(diff_df.to_string(index=False))
            # Топ-5 дат с максимальным расхождением (по модулю diff)
            top5 = diff_df.nlargest(5, 'diff')[['date', 'diff']]
            total_diff = diff_df['diff'].sum()
            print("\n=== Топ-5 дат с максимальным расхождением ===")
            for _, row in top5.iterrows():
                print(f"{row['date'].date()}: {row['diff']}")
            print(f"\nВсего дат с расхождением: {len(diff_df)}, Кол-во расхождений: {total_diff}")

=== Даты с расхождениями ===


=== Топ-5 дат с максимальным расхождением ===
2026-02-02: 478
2025-03-21: 16
2026-04-01: 12
2024-07-09: 6
2023-10-27: 5

Всего дат с расхождением: 128, Кол-во расхождений: 665
